# Customer Transaction Analytics and Anomaly Detection

## 1. Data Collection and Cleaning

### Project Overview
This project analyses customer banking transactions to understand
transaction patterns, customer behaviour, and potentially unusual
financial activities.

### Objectives
- Load and understand the transaction dataset.
- Inspect data quality and missing values.
- Clean inconsistent column names and categorical values.
- Standardize transaction dates.
- Prepare the dataset for exploratory data analysis and modelling.

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

In [15]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "payment_transactions.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (12043, 23)


,status,card_present_flag,bpay_biller_code,account,currency,long_lat,txn_description,merchant_id,merchant_code,first_name,balance,date,gender,age,merchant_suburb,merchant_state,extraction,amount,transaction_id,country,customer_id,merchant_long_lat,movement
0,authorized,1.0,NaN,ACC-1598451071,AUD,153.41 -27.95,POS,81c48296-73be-44a7-befa-d053f48ce7cd,NaN,Diana,35.39,08-01-2018,F,26,Ashmore,QLD,2018-08-01T01:01:15.000+0000,16.25,a623070bfead4541a6b0fff8a09e706c,Australia,CUS-2487424745,153.38 -27.99,debit
1,authorized,0.0,NaN,ACC-1598451071,AUD,153.41 -27.95,SALES-POS,830a451c-316e-4a6a-bf25-e37caedca49e,NaN,Diana,21.20,08-01-2018,F,26,Sydney,NSW,2018-08-01T01:13:45.000+0000,14.19,13270a2a902145da9db4c951e04b51b9,Australia,CUS-2487424745,151.21 -33.87,debit
2,authorized,1.0,NaN,ACC-1222300524,AUD,151.23 -33.94,POS,835c231d-8cdf-4e96-859d-e9d571760cf0,NaN,Michael,5.71,08-01-2018,M,38,Sydney,NSW,2018-08-01T01:26:15.000+0000,6.42,feb79e7ecd7048a5a36ec889d1a94270,Australia,CUS-2142601169,151.21 -33.87,debit
3,authorized,1.0,NaN,ACC-1037050564,AUD,153.10 -27.66,SALES-POS,48514682-c78a-4a88-b0da-2d6302e64673,NaN,Rhonda,2117.22,08-01-2018,F,40,Buderim,QLD,2018-08-01T01:38:45.000+0000,40.90,2698170da3704fd981b15e64a006079e,Australia,CUS-1614226872,153.05 -26.68,debit
4,authorized,1.0,NaN,ACC-1598451071,AUD,153.41 -27.95,SALES-POS,b4e02c10-0852-4273-b8fd-7b3395e32eb0,NaN,Diana,17.95,08-01-2018,F,26,Mermaid Beach,QLD,2018-08-01T01:51:15.000+0000,3.25,329adf79878c4cf0aeb4188b4691c266,Australia,CUS-2487424745,153.44 -28.06,debit


In [16]:
print("Dataset dimensions:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

Dataset dimensions: (12043, 23)

Column names:
['    status', 'card_present_flag', 'bpay_biller_code', 'account', 'currency', 'long_lat', 'txn_description', 'merchant_id', 'merchant_code', 'first_name', 'balance', 'date', 'gender', 'age', 'merchant_suburb', 'merchant_state', 'extraction', 'amount', 'transaction_id', 'country', 'customer_id', 'merchant_long_lat', 'movement']

Data types:
    status               str
card_present_flag    float64
bpay_biller_code         str
account                  str
currency                 str
long_lat                 str
txn_description          str
merchant_id              str
merchant_code        float64
first_name               str
balance              float64
date                     str
gender                   str
age                    int64
merchant_suburb          str
merchant_state           str
extraction               str
amount               float64
transaction_id           str
country                  str
customer_id              str
m

In [17]:
missing_values = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})

missing_values = missing_values[
    missing_values["Missing Values"] > 0
].sort_values(by="Missing Percentage", ascending=False)

display(missing_values)

print("Duplicate rows:", df.duplicated().sum())

,Missing Values,Missing Percentage
merchant_code,11160,92.67
bpay_biller_code,11158,92.65
card_present_flag,4326,35.92
merchant_id,4326,35.92
merchant_suburb,4326,35.92
merchant_state,4326,35.92
merchant_long_lat,4326,35.92


Duplicate rows: 0


In [18]:
# Remove unnecessary leading and trailing spaces from column names
df.columns = df.columns.str.strip()

print("Column names after cleaning:")
print(df.columns.tolist())

Column names after cleaning:
['status', 'card_present_flag', 'bpay_biller_code', 'account', 'currency', 'long_lat', 'txn_description', 'merchant_id', 'merchant_code', 'first_name', 'balance', 'date', 'gender', 'age', 'merchant_suburb', 'merchant_state', 'extraction', 'amount', 'transaction_id', 'country', 'customer_id', 'merchant_long_lat', 'movement']


In [19]:
# Remove unnecessary spaces from text columns
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    df[col] = df[col].str.strip()

print("Categorical values cleaned successfully.")

print("\nStatus categories:")
print(df["status"].value_counts())

print("\nTransaction descriptions:")
print(df["txn_description"].value_counts())

Categorical values cleaned successfully.

Status categories:
status
authorized    7717
posted        4326
Name: count, dtype: int64

Transaction descriptions:
txn_description
SALES-POS     3934
POS           3783
PAYMENT       2600
PAY/SALARY     883
INTER BANK     742
PHONE BANK     101
Name: count, dtype: int64


C:\Users\laksh\AppData\Local\Temp\ipykernel_23496\1078010681.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


In [20]:
# Remove spaces from date values
dates = df["date"].astype("string").str.strip()

# Identify the two date formats
dash_mask = dates.str.contains("-", na=False)
slash_mask = dates.str.contains("/", na=False)

# Create a properly formatted date column
df["date_clean"] = pd.NaT

df.loc[dash_mask, "date_clean"] = pd.to_datetime(
    dates[dash_mask],
    format="%m-%d-%Y",
    errors="coerce"
)

df.loc[slash_mask, "date_clean"] = pd.to_datetime(
    dates[slash_mask],
    format="%m/%d/%Y",
    errors="coerce"
)

print("Total records:", len(df))
print("Successfully converted:", df["date_clean"].notna().sum())
print("Invalid dates:", df["date_clean"].isna().sum())

print("\nDate range:")
print(df["date_clean"].min(), "to", df["date_clean"].max())

Total records: 12043
Successfully converted: 12043
Invalid dates: 0

Date range:
2018-08-01 00:00:00 to 2018-10-31 00:00:00


In [21]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate records:", df.duplicated().sum())

print("\nCleaned dataset preview:")
display(df.head())

Dataset shape: (12043, 24)

Missing values:


status                   0
card_present_flag     4326
bpay_biller_code     11158
account                  0
currency                 0
long_lat                 0
txn_description          0
merchant_id           4326
merchant_code        11160
first_name               0
balance                  0
date                     0
gender                   0
age                      0
merchant_suburb       4326
merchant_state        4326
extraction               0
amount                   0
transaction_id           0
country                  0
customer_id              0
merchant_long_lat     4326
movement                 0
date_clean               0
dtype: int64


Duplicate records: 0

Cleaned dataset preview:


,status,card_present_flag,bpay_biller_code,account,currency,long_lat,txn_description,merchant_id,merchant_code,first_name,balance,date,gender,age,merchant_suburb,merchant_state,extraction,amount,transaction_id,country,customer_id,merchant_long_lat,movement,date_clean
0,authorized,1.0,NaN,ACC-1598451071,AUD,153.41 -27.95,POS,81c48296-73be-44a7-befa-d053f48ce7cd,NaN,Diana,35.39,08-01-2018,F,26,Ashmore,QLD,2018-08-01T01:01:15.000+0000,16.25,a623070bfead4541a6b0fff8a09e706c,Australia,CUS-2487424745,153.38 -27.99,debit,2018-08-01
1,authorized,0.0,NaN,ACC-1598451071,AUD,153.41 -27.95,SALES-POS,830a451c-316e-4a6a-bf25-e37caedca49e,NaN,Diana,21.20,08-01-2018,F,26,Sydney,NSW,2018-08-01T01:13:45.000+0000,14.19,13270a2a902145da9db4c951e04b51b9,Australia,CUS-2487424745,151.21 -33.87,debit,2018-08-01
2,authorized,1.0,NaN,ACC-1222300524,AUD,151.23 -33.94,POS,835c231d-8cdf-4e96-859d-e9d571760cf0,NaN,Michael,5.71,08-01-2018,M,38,Sydney,NSW,2018-08-01T01:26:15.000+0000,6.42,feb79e7ecd7048a5a36ec889d1a94270,Australia,CUS-2142601169,151.21 -33.87,debit,2018-08-01
3,authorized,1.0,NaN,ACC-1037050564,AUD,153.10 -27.66,SALES-POS,48514682-c78a-4a88-b0da-2d6302e64673,NaN,Rhonda,2117.22,08-01-2018,F,40,Buderim,QLD,2018-08-01T01:38:45.000+0000,40.90,2698170da3704fd981b15e64a006079e,Australia,CUS-1614226872,153.05 -26.68,debit,2018-08-01
4,authorized,1.0,NaN,ACC-1598451071,AUD,153.41 -27.95,SALES-POS,b4e02c10-0852-4273-b8fd-7b3395e32eb0,NaN,Diana,17.95,08-01-2018,F,26,Mermaid Beach,QLD,2018-08-01T01:51:15.000+0000,3.25,329adf79878c4cf0aeb4188b4691c266,Australia,CUS-2487424745,153.44 -28.06,debit,2018-08-01


In [22]:
#cell 10
df = df.rename(columns={
    "date": "date_original",
    "date_clean": "date"
})

df["extraction"] = pd.to_datetime(
    df["extraction"],
    errors="coerce",
    utc=True
)

print("Date columns standardized.")
print(df[["date_original", "date", "extraction"]].dtypes)

Date columns standardized.
date_original                    str
date                  datetime64[ns]
extraction       datetime64[us, UTC]
dtype: object


In [23]:
# cell 11
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate records:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Dataset shape: (12043, 24)

Missing values:
status                   0
card_present_flag     4326
bpay_biller_code     11158
account                  0
currency                 0
long_lat                 0
txn_description          0
merchant_id           4326
merchant_code        11160
first_name               0
balance                  0
date_original            0
gender                   0
age                      0
merchant_suburb       4326
merchant_state        4326
extraction               0
amount                   0
transaction_id           0
country                  0
customer_id              0
merchant_long_lat     4326
movement                 0
date                     0
dtype: int64

Duplicate records: 0

Data types:
status                               str
card_present_flag                float64
bpay_biller_code                     str
account                              str
currency                             str
long_lat                             str
txn_descriptio

In [24]:
# cell 12
# Create processed folder if it doesn't exist
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# Save cleaned dataset
OUTPUT_PATH = PROCESSED_PATH / "cleaned_transactions.csv"

df.to_csv(OUTPUT_PATH, index=False)

print("Cleaned dataset saved successfully!")
print("Location:", OUTPUT_PATH)
print("Records saved:", len(df))

Cleaned dataset saved successfully!
Location: c:\Users\laksh\OneDrive\Desktop\projects\customer_transaction_prediction_analysis\data\processed\cleaned_transactions.csv
Records saved: 12043
